## Initialization

In [1]:
from torch.func import vjp
import torch
from torch.func import jacrev, functional_call
import torch.nn as nn
from torch import Tensor

import torch.nn.functional as F

import sys
import os

current_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root_dir = os.path.abspath(os.path.join(current_notebook_dir, '../../'))

# 将这个父目录添加到sys.path的最前面
if project_root_dir not in sys.path:
    sys.path.insert(0, project_root_dir)

print(sys.path)

['/home/hqdeng7/lijuyang/generalization', '/home/hqdeng7/.conda/envs/ljy/lib/python311.zip', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11/lib-dynload', '', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11/site-packages']


In [2]:
from loss_distribution.pytorch_script.visual_utils \
	import load_cifar10_data, load_model_state_dict

from ntk_result.trials.utils import *

import torchvision
import torchvision.transforms as transforms

data_pth = '/home/hqdeng7/lijuyang/generalization/loss_distribution/pytorch_script/data/cifar10'
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
train_ds = torchvision.datasets.CIFAR10(root=data_pth, train=True, download=True, transform=transform)
test_ds = torchvision.datasets.CIFAR10(root=data_pth, train=False, download=True, transform=transform)

In [3]:
model200_path = '/home/hqdeng7/lijuyang/generalization/loss_distribution/model_training_results/cifar10_resnet20/model_200.pth'
model200 = load_model_state_dict('cifar10', 'resnet20', 10, model200_path, 'cuda')
model100_path = '/home/hqdeng7/lijuyang/generalization/loss_distribution/model_training_results/cifar10_resnet20/model_100.pth'
model100 = load_model_state_dict('cifar10', 'resnet20', 10, model100_path, 'cuda')

  从字典中提取模型状态字典...
提取成功
  从字典中提取模型状态字典...
提取成功


## continueTrain

### one cluster continue

In [4]:
test_indices = np.load('cls30_indices_arrs.npz')['arr0']
train_indices = np.load('train_indices.npy')

In [5]:
from torch.utils.data import Subset, DataLoader

train_dl = DataLoader(train_ds, batch_size=512)
test_dl = DataLoader(test_ds, batch_size=512)

train_subds = Subset(train_ds, train_indices)
train_subdl = DataLoader(train_subds, batch_size=512)

test_subds = Subset(test_ds, test_indices)
test_subdl = DataLoader(test_subds, batch_size=512)

In [6]:
def evaluate_model(model, dataloader, device='cuda'):
	model.to(device)
	model.eval()
	total_loss = 0.0
	correct = 0
	total = 0
	criterion = nn.CrossEntropyLoss()
	with torch.no_grad():
		for inputs, targets in dataloader:
			inputs, targets = inputs.to(device), targets.to(device)
			outputs = model(inputs)
			loss = criterion(outputs, targets)
			total_loss += loss.item() * inputs.size(0)
			_, predicted = outputs.max(1)
			correct += predicted.eq(targets).sum().item()
			total += targets.size(0)
	avg_loss = total_loss / total
	acc = correct / total
	return avg_loss, acc


In [7]:
train_loss0, train_acc0 = evaluate_model(model100, train_dl)
test_loss0, test_acc0 = evaluate_model(model100, test_dl)
test_sub_loss0, test_sub_acc0 = evaluate_model(model100, test_subdl)

In [8]:
print("Original loss/acc")
print(f"train loss: {train_loss0:.4f}, train acc: {train_acc0:.4f}")
print(f"test loss: {test_loss0:.4f}, test acc: {test_acc0:.4f}")
print(f"sub test loss: {test_sub_loss0:.4f}, sub test acc: {test_sub_acc0:.4f}")

Original loss/acc
train loss: 0.1778, train acc: 0.9397
test loss: 0.2901, test acc: 0.9018
sub test loss: 3.4284, sub test acc: 0.0000


In [21]:
from torch.optim import SGD

def train_one_batch(model, batch, optimizer):
	model.train()
	criterion = nn.CrossEntropyLoss()

	inputs, targets = batch
	inputs, targets = inputs.cuda(), targets.cuda()

	optimizer.zero_grad()
	outputs = model(inputs)
	loss = criterion(outputs, targets)
	loss.backward()
	optimizer.step()

	print(f"One training step done. Loss: {loss.item():.4f}")

In [ ]:
from loss_distribution.pytorch_script.models import get_model
new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.1
optimizer = torch.optim.SGD(new_model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
train_one_batch(new_model, next(iter(train_subdl)), optimizer)

One training step done. Loss: 4.0950


In [13]:
print(evaluate_model(model100, train_subdl))

(0.47641685605049133, 0.838)


In [15]:
train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

In [16]:
print("After training one cls30 cluster")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")
print(f"sub test loss: {test_sub_loss1:.4f}, sub test acc: {test_sub_acc1:.4f}")

After training one cls30 cluster
lr: 0.1
train loss: 0.4836, train acc: 0.8364
test loss: 0.6236, test acc: 0.7998
sub test loss: 0.4169, sub test acc: 0.8649


In [17]:
print("Original loss/acc")
print(f"train loss: {train_loss0:.4f}, train acc: {train_acc0:.4f}")
print(f"test loss: {test_loss0:.4f}, test acc: {test_acc0:.4f}")
print(f"sub test loss: {test_sub_loss0:.4f}, sub test acc: {test_sub_acc0:.4f}")

Original loss/acc
train loss: 0.1778, train acc: 0.9397
test loss: 0.2901, test acc: 0.9018
sub test loss: 3.4284, sub test acc: 0.0000


### one cluster + random

In [ ]:
train_sub_indices = np.reshape(train_indices_arrs, (-1,))
train_subds = Subset(train_ds, train_sub_indices)
train_subds = Subset(train_ds, train_indices)

test_subds = Subset(test_ds, test_indices)
test_subdl = DataLoader(test_subds, batch_size=512)

In [19]:
orig_indices = train_subds.indices
num_random = len(orig_indices) * 10
random_indices = np.random.choice(np.array(range(len(train_ds))), size=num_random, replace=False)

combined_indices = np.concatenate([orig_indices, random_indices])

bigger_train_subds = Subset(train_ds, combined_indices)
bigger_train_subdl = DataLoader(bigger_train_subds, batch_size=512, shuffle=True)

In [22]:
new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.05
optimizer = torch.optim.SGD(new_model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

for batch in bigger_train_subdl:
	train_one_batch(new_model, batch, optimizer)

One training step done. Loss: 0.3238
One training step done. Loss: 0.2823


One training step done. Loss: 0.3086
One training step done. Loss: 0.2764
One training step done. Loss: 0.2146
One training step done. Loss: 0.1990
One training step done. Loss: 0.1894
One training step done. Loss: 0.1876
One training step done. Loss: 0.2133
One training step done. Loss: 0.1972
One training step done. Loss: 0.1652


In [23]:
train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

In [34]:
print("After training one cls30 + random")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")
print(f"sub test loss: {test_sub_loss1:.4f}, sub test acc: {test_sub_acc1:.4f}")

After training one cls30 + random
lr: 0.05
train loss: 0.1816, train acc: 0.9384
test loss: 0.3052, test acc: 0.8949
sub test loss: 2.2286, sub test acc: 0.2771


### cls10 one cluster + random

In [35]:
test_indices = np.load('cls10_indices_arrs.npz', allow_pickle=True)['arr_0']
train_indices = np.load('cls10_train_indices_arrs.npy')[0]

In [36]:
train_subds = Subset(train_ds, train_indices)

test_subds = Subset(test_ds, test_indices)
test_subdl = DataLoader(test_subds, batch_size=512)

In [37]:
orig_indices = train_subds.indices
num_random = len(orig_indices) * 10
random_indices = np.random.choice(np.array(range(len(train_ds))), size=num_random, replace=False)

combined_indices = np.concatenate([orig_indices, random_indices])

bigger_train_subds = Subset(train_ds, combined_indices)
bigger_train_subdl = DataLoader(bigger_train_subds, batch_size=512, shuffle=True)

In [38]:
new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.05
optimizer = torch.optim.SGD(new_model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

for batch in bigger_train_subdl:
	train_one_batch(new_model, batch, optimizer)

for batch in bigger_train_subdl:
	train_one_batch(new_model, batch, optimizer)

One training step done. Loss: 0.3584
One training step done. Loss: 0.2966
One training step done. Loss: 0.2622
One training step done. Loss: 0.1824
One training step done. Loss: 0.2116
One training step done. Loss: 0.1988
One training step done. Loss: 0.2312
One training step done. Loss: 0.2495
One training step done. Loss: 0.1973
One training step done. Loss: 0.2017
One training step done. Loss: 0.1875
One training step done. Loss: 0.1498
One training step done. Loss: 0.1223
One training step done. Loss: 0.1415
One training step done. Loss: 0.1624
One training step done. Loss: 0.1278
One training step done. Loss: 0.1361
One training step done. Loss: 0.1425
One training step done. Loss: 0.1151
One training step done. Loss: 0.1477
One training step done. Loss: 0.1357
One training step done. Loss: 0.1372


In [39]:
train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

In [42]:
print("After trainig one cls10 cluster + random samples")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")
print(f"sub test loss: {test_sub_loss1:.4f}, sub test acc: {test_sub_acc1:.4f}")

After trainig one cls10 cluster + random samples
lr: 0.05
train loss: 0.1873, train acc: 0.9370
test loss: 0.3119, test acc: 0.8941
sub test loss: 2.1603, sub test acc: 0.3133


In [43]:
test_sub_loss0, test_sub_acc0 = evaluate_model(model100, test_subdl)
print("Original loss/acc")
print(f"train loss: {train_loss0:.4f}, train acc: {train_acc0:.4f}")
print(f"test loss: {test_loss0:.4f}, test acc: {test_acc0:.4f}")
print(f"sub test loss: {test_sub_loss0:.4f}, sub test acc: {test_sub_acc0:.4f}")

Original loss/acc
train loss: 0.1778, train acc: 0.9397
test loss: 0.2901, test acc: 0.9018
sub test loss: 3.2675, sub test acc: 0.0000


### all clusters continue

#### preparation

In [72]:
data = np.load("filtered_cls30_indices_arrs.npz", allow_pickle=True)
filtered_cls30_indices_arrs = [data[f"arr_{i}"] for i in range(len(data.files))]

In [45]:
train_indices_arrs = np.load('train_indices_arrs.npy')

In [46]:
from torch.utils.data import Subset, DataLoader

test_sub_indices = np.concatenate(filtered_cls30_indices_arrs)
test_subds = Subset(test_ds, test_sub_indices)
test_subdl = DataLoader(test_subds, batch_size=512)

In [47]:
test_sub_loss0, test_sub_acc0 = evaluate_model(model100, test_subdl)
print("Original loss/acc")
print(f"train loss: {train_loss0:.4f}, train acc: {train_acc0:.4f}")
print(f"test loss: {test_loss0:.4f}, test acc: {test_acc0:.4f}")
print(f"sub test loss: {test_sub_loss0:.4f}, sub test acc: {test_sub_acc0:.4f}")

Original loss/acc
train loss: 0.1778, train acc: 0.9397
test loss: 0.2901, test acc: 0.9018
sub test loss: 3.2753, sub test acc: 0.0000


#### in order

In [51]:
new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.05
optimizer = torch.optim.SGD(new_model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
for train_indices in train_indices_arrs:
	train_subds = Subset(train_ds, train_indices)
	train_subdl = DataLoader(train_subds, batch_size=512)
	train_one_batch(new_model, next(iter(train_subdl)), optimizer)


One training step done. Loss: 4.0950
One training step done. Loss: 3.0241
One training step done. Loss: 2.6805
One training step done. Loss: 5.1013
One training step done. Loss: 3.3062
One training step done. Loss: 1.0874
One training step done. Loss: 7.9019
One training step done. Loss: 8.2039
One training step done. Loss: 6.8446
One training step done. Loss: 0.9077
One training step done. Loss: 5.6090
One training step done. Loss: 5.2087
One training step done. Loss: 3.9254
One training step done. Loss: 3.0801
One training step done. Loss: 2.9794
One training step done. Loss: 2.0709
One training step done. Loss: 2.7259
One training step done. Loss: 4.7180
One training step done. Loss: 2.6979
One training step done. Loss: 4.9828
One training step done. Loss: 2.9228
One training step done. Loss: 3.1328
One training step done. Loss: 1.8911
One training step done. Loss: 1.8013


In [52]:
train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

In [53]:
print("After training clusters in order")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")
print(f"sub test loss: {test_sub_loss1:.4f}, sub test acc: {test_sub_acc1:.4f}")

After training clusters in order
lr: 0.05
train loss: 2.9038, train acc: 0.1934
test loss: 2.9517, test acc: 0.1885
sub test loss: 3.2010, sub test acc: 0.1266


In [54]:
import torch
torch.cuda.empty_cache()

#### shuffle

In [55]:
import random

seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed) 
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [56]:
train_sub_indices = np.reshape(train_indices_arrs, (-1,))
train_subds = Subset(train_ds, train_sub_indices)
train_subdl = DataLoader(train_subds, batch_size=512, shuffle=True)

new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.05
optimizer = torch.optim.SGD(new_model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
for batch in train_subdl:
	train_one_batch(new_model, batch, optimizer)


One training step done. Loss: 0.5578
One training step done. Loss: 0.4606


One training step done. Loss: 0.4173
One training step done. Loss: 0.3224
One training step done. Loss: 0.3093
One training step done. Loss: 0.3048
One training step done. Loss: 0.2871
One training step done. Loss: 0.3509
One training step done. Loss: 0.3051
One training step done. Loss: 0.3577
One training step done. Loss: 0.2521
One training step done. Loss: 0.2896
One training step done. Loss: 0.3007
One training step done. Loss: 0.2514
One training step done. Loss: 0.2838
One training step done. Loss: 0.2970
One training step done. Loss: 0.2618
One training step done. Loss: 0.2746
One training step done. Loss: 0.2463
One training step done. Loss: 0.2794
One training step done. Loss: 0.2220
One training step done. Loss: 0.2688
One training step done. Loss: 0.2688
One training step done. Loss: 0.2262


In [57]:
train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

In [58]:
print("After training clusters with shuffle")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")
print(f"sub test loss: {test_sub_loss1:.4f}, sub test acc: {test_sub_acc1:.4f}")

After training clusters with shuffle
lr: 0.05
train loss: 0.2235, train acc: 0.9242
test loss: 0.3506, test acc: 0.8801
sub test loss: 2.6435, sub test acc: 0.1709


#### all clusters + random

In [59]:
import numpy as np
from torch.utils.data import ConcatDataset, Subset

orig_indices = train_subds.indices
num_random = len(orig_indices) * 4
random_indices = np.random.choice(np.array(range(len(train_ds))), size=num_random, replace=False)

combined_indices = np.concatenate([orig_indices, random_indices])
bigger_train_subds = Subset(train_ds, combined_indices)
bigger_train_subdl = DataLoader(bigger_train_subds, batch_size=512, shuffle=True)

In [60]:
len(train_ds), len(orig_indices)

(50000, 12000)

In [78]:
import io
import contextlib


new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.05
optimizer = torch.optim.SGD(new_model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
for batch in bigger_train_subdl:
	with contextlib.redirect_stdout(io.StringIO()):
		train_one_batch(new_model, batch, optimizer)

# for batch in bigger_train_subdl:
# 	train_one_batch(new_model, batch, optimizer)

In [79]:
train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

In [80]:
print("After training shuffled cls30 + random samples")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")
print(f"sub test loss: {test_sub_loss1:.4f}, sub test acc: {test_sub_acc1:.4f}")

After training shuffled cls30 + random samples
lr: 0.05
train loss: 0.1379, train acc: 0.9571
test loss: 0.2755, test acc: 0.9055
sub test loss: 2.8751, sub test acc: 0.0361


#### cls15 + random

In [74]:
test_indices = np.load('cls10_indices_arrs.npz', allow_pickle=True)['arr_0']
train_indices_arrs = np.load('cls10_train_indices_arrs.npy')
train_indices = np.reshape(train_indices_arrs, (-1,))

train_subds = Subset(train_ds, train_indices)
train_subdl = DataLoader(train_subds, batch_size=512, shuffle=True)

orig_indices = train_subds.indices
num_random = len(orig_indices) * 6
random_indices = np.random.choice(np.array(range(len(train_ds))), size=num_random, replace=False)

combined_indices = np.concatenate([orig_indices, random_indices])
bigger_train_subds = Subset(train_ds, combined_indices)
bigger_train_subdl = DataLoader(bigger_train_subds, batch_size=512, shuffle=True)

new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.05
optimizer = torch.optim.SGD(new_model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
for batch in bigger_train_subdl:
	train_one_batch(new_model, batch, optimizer)

for batch in bigger_train_subdl:
	train_one_batch(new_model, batch, optimizer)

train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

print("After training shuffled cls15 + random samples")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")
print(f"sub test loss: {test_sub_loss1:.4f}, sub test acc: {test_sub_acc1:.4f}")

One training step done. Loss: 0.2408
One training step done. Loss: 0.3411


One training step done. Loss: 0.2870
One training step done. Loss: 0.2495
One training step done. Loss: 0.2249
One training step done. Loss: 0.1700
One training step done. Loss: 0.2386
One training step done. Loss: 0.2227
One training step done. Loss: 0.1794
One training step done. Loss: 0.2235
One training step done. Loss: 0.2019
One training step done. Loss: 0.2175
One training step done. Loss: 0.1687
One training step done. Loss: 0.1856
One training step done. Loss: 0.2215
One training step done. Loss: 0.1973
One training step done. Loss: 0.1360
One training step done. Loss: 0.1953
One training step done. Loss: 0.1827
One training step done. Loss: 0.2057
One training step done. Loss: 0.1744
One training step done. Loss: 0.1775
One training step done. Loss: 0.1841
One training step done. Loss: 0.1764
One training step done. Loss: 0.1901
One training step done. Loss: 0.1527
One training step done. Loss: 0.1905
One training step done. Loss: 0.1367
One training step done. Loss: 0.1686
O

## contrast

### one epoch

In [ ]:
import io
import contextlib

new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.05
optimizer = SGD(new_model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

for batch in train_dl:
	with contextlib.redirect_stdout(io.StringIO()):
		train_one_batch(new_model, batch, optimizer)

train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

print("After training one epoch")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")
print(f"sub test loss: {test_sub_loss1:.4f}, sub test acc: {test_sub_acc1:.4f}")

After training one epoch
lr: 0.05
train loss: 0.1270, train acc: 0.9600
test loss: 0.2708, test acc: 0.9052
sub test loss: 2.5297, sub test acc: 0.1688


#### 5 epochs

In [75]:
import io
import contextlib

new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.05
optimizer = SGD(new_model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

for i in range(5):
	for batch in train_dl:
		with contextlib.redirect_stdout(io.StringIO()):
			train_one_batch(new_model, batch, optimizer)

train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

print("After training one epoch")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")
print(f"sub test loss: {test_sub_loss1:.4f}, sub test acc: {test_sub_acc1:.4f}")

After training one epoch
lr: 0.05
train loss: 0.0313, train acc: 0.9938
test loss: 0.3138, test acc: 0.9075
sub test loss: 2.9342, sub test acc: 0.2289


In [ ]:
test_indices = np.load('cls10_indices_arrs.npz', allow_pickle=True)['arr_0']
train_indices_arrs = np.load('cls30_train_indices_arrs.npy')
train_indices = np.reshape(train_indices_arrs, (-1,))

train_subds = Subset(train_ds, train_indices)
train_subdl = DataLoader(train_subds, batch_size=512, shuffle=True)

orig_indices = train_subds.indices
num_random = len(orig_indices) * 6
random_indices = np.random.choice(np.array(range(len(train_ds))), size=num_random, replace=False)

combined_indices = np.concatenate([orig_indices, random_indices])
bigger_train_subds = Subset(train_ds, combined_indices)
bigger_train_subdl = DataLoader(bigger_train_subds, batch_size=512, shuffle=True)

new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.05
optimizer = torch.optim.SGD(new_model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
for i in range(5):
	for batch in bigger_train_subdl:
		with contextlib.redirect_stdout(io.StringIO()):
			train_one_batch(new_model, batch, optimizer)

train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

print("After training shuffled cls15 + random samples")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")
print(f"sub test loss: {test_sub_loss1:.4f}, sub test acc: {test_sub_acc1:.4f}")

After training shuffled cls15 + random samples
lr: 0.05
train loss: 0.0650, train acc: 0.9822
test loss: 0.2937, test acc: 0.9068
sub test loss: 3.5131, sub test acc: 0.0964


### naive hard example mining

In [66]:
from torch.utils.data import Subset, DataLoader

# Find top 1000 high-loss samples from the training set
topk = 1000
batch_loss_fn = get_batch_loss_fn(model100)
high_loss_indices = list(zip(*find_topk_samples(train_ds, batch_loss_fn, k=topk)))[1]

# Create DataLoader for high-loss samples
high_loss_subds = Subset(train_ds, high_loss_indices)
high_loss_subdl = DataLoader(high_loss_subds, batch_size=512, shuffle=True)

100%|██████████| 196/196 [00:13<00:00, 14.03it/s]


In [67]:
# Re-initialize new_model from model100
new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.05
for batch in high_loss_subdl:
	train_one_batch(new_model, batch, optimizer)

# Evaluate and print results
train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

print("After training on high-loss samples")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")

One training step done. Loss: 3.5379
One training step done. Loss: 3.6107
After training on high-loss samples
lr: 0.05
train loss: 0.1772, train acc: 0.9392
test loss: 0.2932, test acc: 0.9001


In [68]:
len(high_loss_subdl)

2

### online hard example mining

In [69]:
import torch.nn.functional as F

def train_one_epoch_ohem(model, dataloader, optimizer, device, top_k_ratio=0.5):
    """
    Args:
        model: nn.Module
        dataloader: DataLoader
        optimizer: torch.optim
        device: 'cuda' or 'cpu'
        top_k_ratio: 选取 batch 内 top-k 样本的比例 (0~1)
    """
    model.train()
    criterion = nn.CrossEntropyLoss(reduction='none')  # 不要平均，返回每个样本的 loss

    total_loss, total_correct, total_seen = 0, 0, 0

    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        # 逐样本 loss
        losses = criterion(outputs, targets)  # shape: (batch_size,)
        
        # 选取 batch 内 top-k
        batch_size = losses.shape[0]
        k = max(1, int(batch_size * top_k_ratio))  # 至少要有1个
        topk_loss, topk_idx = torch.topk(losses, k)

        # 只对 hardest k 个样本反传
        topk_loss.mean().backward()
        optimizer.step()

        # 统计
        preds = outputs.argmax(dim=1)
        total_correct += (preds == targets).sum().item()
        total_seen += batch_size
        total_loss += topk_loss.sum().item()  # 累加 hard 样本 loss

    avg_loss = total_loss / total_seen
    avg_acc = total_correct / total_seen
    return avg_loss, avg_acc

In [70]:
new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')

lr = 0.05
optimizer = torch.optim.SGD(new_model.parameters(), lr=lr, momentum=0.9)
train_one_epoch_ohem(new_model, train_dl, optimizer, "cuda", top_k_ratio=1)

# Evaluate and print results
train_loss1, train_acc1 = evaluate_model(new_model, train_dl)
test_loss1, test_acc1 = evaluate_model(new_model, test_dl)
test_sub_loss1, test_sub_acc1 = evaluate_model(new_model, test_subdl)

print("After online hard example mining")
print(f"lr: {lr}")
print(f"train loss: {train_loss1:.4f}, train acc: {train_acc1:.4f}")
print(f"test loss: {test_loss1:.4f}, test acc: {test_acc1:.4f}")

After online hard example mining
lr: 0.05
train loss: 0.1207, train acc: 0.9621
test loss: 0.2703, test acc: 0.9077
